# Diferencias finitas de II.F

Calbuco, conducto cilíndrico. En cada altura el Thomas resuelve el momento de la mezcla en `r` (inercia en la diagonal, pared en 0). `dP/dz` es el que conserva el caudal. `φ` es la fracción de Henry: con arrastre de Stokes las dos masas dan eso y `u_g = u_m`. `v_in = 23 m/s` se ahoga cerca de la boca: abajo el gradiente es el del peso y en los últimos metros se empina.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from RIconduit2D_IIF import marchar

sal = marchar(23.0, n_r=12)
print("pasos", sal["n_pasos"], "|", sal["mensaje"])
print("v en la base [m/s]", sal["vinicial"])
print("z inicial, final [m]", sal["z"][0], sal["z"][-1])
print("P inicial, final [MPa]", sal["P"][0] / 1e6, sal["P"][-1] / 1e6)
print("u media inicial, final [m/s]", sal["um_media"][0], sal["um_media"][-1])
print("Q [kg/s]", sal["Q_objetivo"])
print("φ máximo", sal["phi"].max())
print("ξ inicial, final", sal["xi"][0, 0], sal["xi"][-1, 0])
boca = marchar(18.0, n_r=12)
print("con 18 m/s llega a la boca |", boca["mensaje"], "P", boca["P"][-1] / 1e6, "φ", boca["phi"][-1, 0])


pasos 102 | ahogado
v en la base [m/s] 23.0
z inicial, final [m] -7000.0 -134.432244195574
P inicial, final [MPa] 183.542 2.4864778859563677
u media inicial, final [m/s] 20.90909090909091 183.4648185032264
Q [kg/s] 41421253.49972596
φ máximo 0.9371843720373163
ξ inicial, final 0.25 0.25016167387428573


con 18 m/s llega a la boca | boca P 11.834279812189667 φ 0.7067591725419026


In [2]:
import matplotlib.ticker as ticker

fig, ejes = plt.subplots(2, 2, figsize=(9, 7))
ejes[0, 0].plot(sal["P"] / 1e6, sal["z"] / 1e3)
ejes[0, 0].set_xscale("log")
ejes[0, 0].set_xlim(1, 300)
ejes[0, 0].set_xticks([1, 10, 100])
ejes[0, 0].xaxis.set_major_formatter(ticker.LogFormatterMathtext())
ejes[0, 0].set_xlabel("P [MPa]")
ejes[0, 0].set_ylabel("z [km]")
recorte = sal["z"] > -1500
detalle = ejes[0, 0].inset_axes([0.15, 0.15, 0.45, 0.42])
detalle.plot(sal["P"][recorte] / 1e6, sal["z"][recorte] / 1e3, color="C0")
detalle.set_xscale("log")
detalle.set_xlim(1, 40)
detalle.set_xticks([1, 10])
detalle.xaxis.set_major_formatter(ticker.LogFormatterMathtext())
detalle.set_title("último 1.5 km", fontsize=8)
detalle.tick_params(labelsize=7)
ejes[0, 1].plot(sal["um_media"], sal["z"] / 1e3, label="u_m = u_g")
ejes[0, 1].set_xscale("log")
ejes[0, 1].set_xlim(10, 300)
ejes[0, 1].set_xticks([10, 100])
ejes[0, 1].xaxis.set_major_formatter(ticker.LogFormatterMathtext())
ejes[0, 1].set_xlabel("velocidad media [m/s]")
ejes[0, 1].legend()
phi_eje = np.ma.masked_where(sal["phi"][:, 0] <= 0, sal["phi"][:, 0])
phi_pared = np.ma.masked_where(sal["phi"][:, -2] <= 0, sal["phi"][:, -2])
ejes[1, 0].plot(phi_eje, sal["z"] / 1e3, label="eje")
ejes[1, 0].plot(phi_pared, sal["z"] / 1e3, label="cerca de la pared")
ejes[1, 0].set_xscale("log")
ejes[1, 0].set_xlim(1e-2, 1)
ejes[1, 0].set_xticks([0.01, 0.1, 1])
ejes[1, 0].xaxis.set_major_formatter(ticker.LogFormatterMathtext())
ejes[1, 0].set_xlabel("φ")
ejes[1, 0].set_ylabel("z [km]")
ejes[1, 0].legend()
ejes[1, 1].plot(sal["xi"][:, 0], sal["z"] / 1e3, label="eje")
ejes[1, 1].plot(sal["xi"][:, -2], sal["z"] / 1e3, label="cerca de la pared")
ejes[1, 1].set_xscale("log")
ejes[1, 1].xaxis.set_major_formatter(ticker.LogFormatterSciNotation())
ejes[1, 1].set_xlabel("ξ")
ejes[1, 1].legend()
fig.tight_layout()
fig.savefig("/opt/cursor/artifacts/iif-columnas-log.png", dpi=140)
plt.show()


In [3]:
r = sal["r"]
dentro = r > 0
niveles = [0, len(sal["z"]) // 2, -1]
fig, ejes = plt.subplots(1, 2, figsize=(9, 3.6))
for j in niveles:
    etiqueta = f"z = {sal['z'][j] / 1e3:.2f} km"
    ejes[0].plot(r[dentro], sal["um"][j, dentro], label=etiqueta)
    phi = np.ma.masked_where(sal["phi"][j] <= 0, sal["phi"][j])
    ejes[1].plot(r[dentro], phi[dentro], label=etiqueta)
for ax in ejes:
    ax.set_xscale("log")
    ax.set_xlim(1, 20)
    ax.set_xticks([1, 2, 5, 10])
    ax.xaxis.set_major_formatter(ticker.LogFormatterMathtext())
    ax.set_xlabel("r [m]")
ejes[0].set_ylabel("u_m = u_g [m/s]")
ejes[1].set_ylabel("φ")
ejes[0].legend()
ejes[1].legend()
fig.tight_layout()
fig.savefig("/opt/cursor/artifacts/iif-perfiles-log.png", dpi=140)
plt.show()
